In [ ]:
import numpy as np
import h5py
import xarray as xr
from joblib import Parallel, delayed

def load_ep_flux_data(pr, save_dir):
    f = np.load(f"{save_dir}/epflux_PR{pr}.npz", "r")
    Fj = f["ep1"][:]  # (time, level, lat)
    Fk = f["ep2"][:]
    lat = f["lat"][:]
    pres = f["pres"][:]
    return Fj, Fk, lat, pres

def composite_mean(F, time_idx_array, lags):
    all_lagged_idx = []
    for lag in lags:
        lagged = time_idx_array + lag
        lagged = lagged[lagged < F.shape[0]]
        all_lagged_idx.append(lagged)
    all_lagged_idx = np.concatenate(all_lagged_idx)
    return F[all_lagged_idx].mean(axis=0)

# Configuration
save_stat_dir = "/data92/PeterChang/back_to_master1220/Moist_Dycore"
save_dir = f"{save_stat_dir}/EPflux"
idx_dir = f"{save_stat_dir}/EPflux/New_PC1_time_idx/"

ref_pr = 0
compare_pr_list = [10, 20, 30, 40, 50]
lags_5_20 = np.arange(5*4, 25*4+1, 1)
n_samples = 500
n_jobs = 8  # Adjust based on your CPU

np.random.seed(0)  # Fix random seed for reproducibility

# Load reference (PR0)
Fj0, Fk0, lat, pres = load_ep_flux_data(ref_pr, save_dir)
Ntime0 = Fj0.shape[0]
with h5py.File(f"{idx_dir}/PC1_positive_1std_PR{ref_pr}.h5") as f:
    idx_pos0 = f["time_idx_pos"][:][0]
with h5py.File(f"{idx_dir}/PC1_negative_1std_PR{ref_pr}.h5") as f:
    idx_neg0 = f["time_idx_neg"][:][0]
n_pos0, n_neg0 = len(idx_pos0), len(idx_neg0)

for PR in compare_pr_list:
    print(f"\nComparing PR{PR} against PR{ref_pr}...")

    Fj, Fk, _, _ = load_ep_flux_data(PR, save_dir)
    Ntime = Fj.shape[0]
    with h5py.File(f"{idx_dir}/PC1_positive_1std_PR{PR}.h5") as f:
        idx_pos = f["time_idx_pos"][:][0]
    with h5py.File(f"{idx_dir}/PC1_negative_1std_PR{PR}.h5") as f:
        idx_neg = f["time_idx_neg"][:][0]
    n_pos, n_neg = len(idx_pos), len(idx_neg)

    # Compute real difference between PRi and PR0
    ep1_pos = composite_mean(Fj, idx_pos, lags_5_20)
    ep2_pos = composite_mean(Fk, idx_pos, lags_5_20)
    ep1_neg = composite_mean(Fj, idx_neg, lags_5_20)
    ep2_neg = composite_mean(Fk, idx_neg, lags_5_20)
    diff_phi_real = (ep1_pos - ep1_neg)

    ep1_pos0 = composite_mean(Fj0, idx_pos0, lags_5_20)
    ep2_pos0 = composite_mean(Fk0, idx_pos0, lags_5_20)
    ep1_neg0 = composite_mean(Fj0, idx_neg0, lags_5_20)
    ep2_neg0 = composite_mean(Fk0, idx_neg0, lags_5_20)
    diff_phi_ref = (ep1_pos0 - ep1_neg0)

    real_diff_phi = diff_phi_real - diff_phi_ref
    real_diff_p = (ep2_pos - ep2_neg) - (ep2_pos0 - ep2_neg0)

    def single_monte_carlo_sample(i):
        rand_idx_pos = np.random.choice(Ntime, size=n_pos, replace=False)
        rand_idx_neg = np.random.choice(Ntime, size=n_neg, replace=False)
        rand_idx_pos0 = np.random.choice(Ntime0, size=n_pos0, replace=False)
        rand_idx_neg0 = np.random.choice(Ntime0, size=n_neg0, replace=False)

        rand_ep1_pos = composite_mean(Fj, rand_idx_pos, lags_5_20)
        rand_ep2_pos = composite_mean(Fk, rand_idx_pos, lags_5_20)
        rand_ep1_neg = composite_mean(Fj, rand_idx_neg, lags_5_20)
        rand_ep2_neg = composite_mean(Fk, rand_idx_neg, lags_5_20)

        rand_ep1_pos0 = composite_mean(Fj0, rand_idx_pos0, lags_5_20)
        rand_ep2_pos0 = composite_mean(Fk0, rand_idx_pos0, lags_5_20)
        rand_ep1_neg0 = composite_mean(Fj0, rand_idx_neg0, lags_5_20)
        rand_ep2_neg0 = composite_mean(Fk0, rand_idx_neg0, lags_5_20)

        diff_phi = (rand_ep1_pos - rand_ep1_neg) - (rand_ep1_pos0 - rand_ep1_neg0)
        diff_p = (rand_ep2_pos - rand_ep2_neg) - (rand_ep2_pos0 - rand_ep2_neg0)
        return diff_phi, diff_p

    results = Parallel(n_jobs=n_jobs)(delayed(single_monte_carlo_sample)(i) for i in range(n_samples))
    diff_phi_samples, diff_p_samples = zip(*results)
    diff_phi_samples = np.stack(diff_phi_samples).astype(np.float32)
    diff_p_samples = np.stack(diff_p_samples).astype(np.float32)

    # Save
    ds = xr.Dataset({
        f"real_diff_phi_PR{PR}_vs_PR{ref_pr}": (["level", "lat"], real_diff_phi),
        f"real_diff_p_PR{PR}_vs_PR{ref_pr}":   (["level", "lat"], real_diff_p),
        f"rand_diff_phi_PR{PR}_vs_PR{ref_pr}": (["sample", "level", "lat"], diff_phi_samples),
        f"rand_diff_p_PR{PR}_vs_PR{ref_pr}":   (["sample", "level", "lat"], diff_p_samples),
    }, coords={"sample": np.arange(n_samples), "level": pres, "lat": lat})

    save_path = f"{save_stat_dir}/EPflux/PR{PR}_vs_PR{ref_pr}_EP_diff_stat_test.nc"
    print(f"Saving to {save_path}...")
    ds.to_netcdf(save_path)
    print("✓ Done.")




Comparing PR10 against PR0...
